In [0]:
from pyspark.sql.functions import col, from_unixtime, when, from_utc_timestamp

print("1. Reading Bronze Live Weather...")
bronze_live_weather = spark.table("aviation_project.bronze_live_weather")

In [0]:
# ---------------------------------------------------------
# Part 1: Schema Alignment and Timezone Correction
# ---------------------------------------------------------
print("2. Piercing the String Shield & Enforcing Silver Schema...")

print("2. Enforcing Silver Schema (Native UTC)...")

silver_live_weather = bronze_live_weather.select(
    # Unix time is natively UTC. Just cast and rename.
    from_unixtime(col("api_timestamp").cast("long")).cast("timestamp").alias("weather_timestamp_utc"),
    
    col("airport_code"),
    col("temperature_f").cast("double"),
    col("wind_speed_mph").cast("double"),
    col("rain_1h").cast("double").alias("precipitation_inch"),
    
    when(col("weather_condition") == "Clouds", "Cloudy")
    .when(col("weather_condition").isin("Mist", "Haze", "Smoke", "Dust", "Fog", "Ash", "Squall"), "Fog")
    .otherwise(col("weather_condition")).alias("condition")
)

In [0]:
silver_live_weather.count()

In [0]:
# ---------------------------------------------------------
# Part 2: Deduplication
# ---------------------------------------------------------
silver_live_weather = silver_live_weather.dropDuplicates(["airport_code", "weather_timestamp_utc"])
silver_live_weather = silver_live_weather.dropna(subset=["weather_timestamp_utc"])

In [0]:
display(silver_live_weather.limit(10))

In [0]:
# ---------------------------------------------------------
# Part 3: Write and Optimize
# ---------------------------------------------------------
print("3. Writing to Silver Delta Table...")
silver_live_weather.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("aviation_project.silver_live_weather")

print("   -> Optimizing table layout (Z-Ordering by airport and time)...")
spark.sql("OPTIMIZE aviation_project.silver_live_weather ZORDER BY (airport_code, weather_timestamp_utc)")

final_count = spark.table("aviation_project.silver_live_weather").count()
print(f"4. SUCCESS! Cleaned and aligned {final_count} Live Silver weather records.")